# 03 - Feature Engineering
This notebook prepares model-ready text features with cleaning, stopwords, and n-gram experimentation.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv(Path('..') / 'data' / 'kdrama_list_cleaned.csv')
df = df.rename(columns={'img URL': 'img_URL'})

for col in ['Synopsis', 'Genre', 'Tags']:
    df[col] = df[col].fillna('').astype(str)

df.head()

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in ['Synopsis', 'Genre', 'Tags']:
    df[col] = df[col].map(clean_text)

df['combined_text'] = (df['Synopsis'] + ' ' + df['Genre'] + ' ' + df['Tags']).str.strip()
df[['Title', 'combined_text']].head()

In [ ]:
vectorizer_uni = TfidfVectorizer(stop_words='english', ngram_range=(1, 1), min_df=2)
X_uni = vectorizer_uni.fit_transform(df['combined_text'])

vectorizer_bi = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_bi = vectorizer_bi.fit_transform(df['combined_text'])

print('Unigram shape:', X_uni.shape)
print('Uni+Bigram shape:', X_bi.shape)

In [ ]:
output_path = Path('..') / 'data' / 'kdrama_features.csv'
df.to_csv(output_path, index=False)
output_path